# MNPS Job Classification Likelihood Scorer - Individual Record Analysis
## Google Colab Notebook for Batch Processing with Per-Record Scores

This notebook calculates likelihood scores for **EACH** job classification record, comparing AI model performance to human HR professionals (88-94% baseline).

### Features:
- Individual likelihood scores for each record
- Flexible input file handling (changes each run)
- Static resource files (rarely change)
- Detailed analysis and visualizations
- Export results to CSV/Excel

---

In [ ]:
#@title 1️⃣ Setup and Install Dependencies { display-mode: "form" }
#@markdown Run this cell first to install required packages

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import files
import io
import warnings
warnings.filterwarnings('ignore')

# Set display options for better readability
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 50)

print("✅ Setup complete! Dependencies installed.")
print("=" * 80)

In [ ]:
#@title 2️⃣ Configuration - Set Human Baseline Parameters { display-mode: "form" }
#@markdown Adjust the human baseline accuracy range based on your organization's data

HUMAN_BASELINE_MIN = 88 #@param {type:"slider", min:80, max:95, step:1}
HUMAN_BASELINE_MAX = 94 #@param {type:"slider", min:85, max:99, step:1}
HUMAN_BASELINE_TYPICAL = (HUMAN_BASELINE_MIN + HUMAN_BASELINE_MAX) / 2

#@markdown Component weights for likelihood calculation (must sum to 100%)
WEIGHT_KSAC_SIMILARITY = 40 #@param {type:"slider", min:0, max:100, step:5}
WEIGHT_SALARY_IMPACT = 20 #@param {type:"slider", min:0, max:100, step:5}
WEIGHT_TIME_IMPACT = 10 #@param {type:"slider", min:0, max:100, step:5}
WEIGHT_SUBGROUP = 15 #@param {type:"slider", min:0, max:100, step:5}
WEIGHT_JUSTIFICATION = 15 #@param {type:"slider", min:0, max:100, step:5}

# Normalize weights
total_weight = WEIGHT_KSAC_SIMILARITY + WEIGHT_SALARY_IMPACT + WEIGHT_TIME_IMPACT + WEIGHT_SUBGROUP + WEIGHT_JUSTIFICATION
weights = {
    'similarity': WEIGHT_KSAC_SIMILARITY / total_weight,
    'salary': WEIGHT_SALARY_IMPACT / total_weight,
    'time': WEIGHT_TIME_IMPACT / total_weight,
    'subgroup': WEIGHT_SUBGROUP / total_weight,
    'justification': WEIGHT_JUSTIFICATION / total_weight
}

print(f"Human Baseline Configuration:")
print(f"  • Minimum: {HUMAN_BASELINE_MIN}%")
print(f"  • Maximum: {HUMAN_BASELINE_MAX}%")
print(f"  • Typical: {HUMAN_BASELINE_TYPICAL:.1f}%")
print(f"\nComponent Weights (normalized):")
for component, weight in weights.items():
    print(f"  • {component.capitalize()}: {weight*100:.1f}%")

In [ ]:
#@title 3️⃣ Upload Resource Files (Rarely Change) { display-mode: "form" }
#@markdown Upload the resource files that define your classification framework.
#@markdown These typically remain constant across multiple runs.

print("Please upload the following RESOURCE files:")
print("1. MNPS_KSACs.csv")
print("2. MNPS_Role_Groups_by_KSAC_Similarity_FINAL.csv")
print("3. salary_by_major_role_grouping.csv")
print("4. Time_to_correct_an_error_in_hours.csv")
print("\n⏳ Waiting for file uploads...")

# Upload resource files
uploaded_resources = files.upload()

# Initialize resource dataframes
resource_data = {}

for filename in uploaded_resources.keys():
    df = pd.read_csv(io.BytesIO(uploaded_resources[filename]), encoding='latin-1')
    
    if 'KSAC' in filename:
        resource_data['ksacs'] = df
        print(f"✅ Loaded KSACs: {len(df)} records")
    elif 'Role_Groups' in filename or 'Similarity' in filename:
        resource_data['role_groups'] = df
        print(f"✅ Loaded Role Groups: {len(df)} groups")
    elif 'salary' in filename.lower():
        resource_data['salary_data'] = df
        print(f"✅ Loaded Salary Data: {len(df)} roles")
    elif 'Time' in filename or 'correct' in filename:
        resource_data['time_to_correct'] = df
        print(f"✅ Loaded Time to Correct: {df.loc[0, 'Average']} hours average")

print("\n✅ All resource files loaded successfully!")

In [ ]:
#@title 4️⃣ Upload Input Files (Change Each Run) { display-mode: "form" }
#@markdown Upload the INPUT files for this specific batch analysis.
#@markdown These files will change with each run.

print("Please upload the following INPUT files for this batch:")
print("1. Sample_JDs.csv (or your job descriptions file)")
print("2. Job_Classifications_Batch_[model]_[version].csv")
print("\n⏳ Waiting for file uploads...")

# Upload input files
uploaded_inputs = files.upload()

# Initialize input dataframes
input_data = {}

for filename in uploaded_inputs.keys():
    df = pd.read_csv(io.BytesIO(uploaded_inputs[filename]), encoding='latin-1')
    
    if 'JD' in filename or 'job_desc' in filename.lower() or 'description' in filename.lower():
        input_data['job_descriptions'] = df
        print(f"✅ Loaded Job Descriptions: {len(df)} jobs")
    elif 'Classification' in filename or 'Batch' in filename:
        input_data['classifications'] = df
        print(f"✅ Loaded Classifications: {len(df)} classified records")

print("\n✅ All input files loaded successfully!")

# Display sample of classifications
print("\n📋 Sample of Classifications:")
display_cols = [col for col in ['job_title_original', 'new_job_title', 'major_role_group', 'minor_sub_group'] if col in input_data['classifications'].columns]
print(input_data['classifications'][display_cols].head())

In [ ]:
#@title 5️⃣ Load Core Calculation Functions { display-mode: "form" }
#@markdown These functions calculate the likelihood score for each record

def clean_salary(salary_str):
    """Clean salary string and convert to float"""
    if pd.isna(salary_str):
        return 0
    return float(str(salary_str).replace('$', '').replace(',', '').replace(' ', ''))

def preprocess_salary_data(salary_data):
    """Clean and preprocess salary data"""
    salary_data = salary_data.copy()
    salary_data.columns = salary_data.columns.str.strip()
    
    for col in ['Min Annual Salary', 'Average Annual Salary', 'Max Annual Salary']:
        if col in salary_data.columns:
            salary_data[col] = salary_data[col].apply(clean_salary)
    
    salary_data['Major Role Grouping'] = salary_data['Major Role Grouping'].str.strip()
    return salary_data

def create_role_mappings(role_groups):
    """Create comprehensive role mappings"""
    role_to_group = {}
    group_roles = {}
    group_details = {}
    
    for idx, row in role_groups.iterrows():
        group_name = row['Group Name']
        roles = [r.strip() for r in row['Roles in Group'].split(',')]
        
        for role in roles:
            role_to_group[role.lower()] = group_name
            
        group_roles[group_name] = [r.lower() for r in roles]
        group_details[group_name] = {
            'roles': roles,
            'count': len(roles)
        }
    
    return role_to_group, group_roles, group_details

def calculate_similarity_score(classification, role_to_group, group_roles):
    """Calculate KSAC similarity score"""
    if pd.isna(classification):
        return 0.0, "No classification provided"
    
    classified_role = str(classification).lower().strip()
    classified_group = role_to_group.get(classified_role, None)
    
    if classified_group is None:
        return 0.0, f"Role '{classification}' not in any KSAC group"
    
    return 1.0, f"Classified in '{classified_group}'"

def calculate_salary_impact(classification, salary_data):
    """Calculate salary-based error severity"""
    if pd.isna(classification):
        return 0.0, "Missing classification", 0
    
    classified_salary = salary_data[
        salary_data['Major Role Grouping'].str.lower() == str(classification).lower()
    ]
    
    if classified_salary.empty:
        return 0.5, f"No salary data for '{classification}'", 0
    
    avg_salary = classified_salary['Average Annual Salary'].values[0]
    max_salary_in_data = salary_data['Max Annual Salary'].max()
    min_salary_in_data = salary_data['Min Annual Salary'].min()
    
    # Normalize salary impact
    if max_salary_in_data > min_salary_in_data:
        severity = (avg_salary - min_salary_in_data) / (max_salary_in_data - min_salary_in_data)
    else:
        severity = 0.5
    
    # Invert for scoring
    score = 1.0 - severity
    
    salary_band = (
        "Entry-level" if avg_salary < 40000 else
        "Mid-level" if avg_salary < 60000 else
        "Senior-level" if avg_salary < 80000 else
        "Management" if avg_salary < 100000 else
        "Executive"
    )
    
    return score, f"{salary_band} (${avg_salary:,.0f})", avg_salary

def calculate_likelihood_from_composite(composite_score):
    """Convert composite score to likelihood score using human baseline"""
    
    if composite_score <= 0.2:
        likelihood = composite_score * 5
    elif composite_score <= HUMAN_BASELINE_MIN/100:
        range_size = HUMAN_BASELINE_MIN/100 - 0.2
        position = (composite_score - 0.2) / range_size
        likelihood = 1.0 + (position * 3.0)
    elif composite_score <= HUMAN_BASELINE_TYPICAL/100:
        range_size = HUMAN_BASELINE_TYPICAL/100 - HUMAN_BASELINE_MIN/100
        position = (composite_score - HUMAN_BASELINE_MIN/100) / range_size
        likelihood = 4.0 + (position * 0.5)
    elif composite_score <= HUMAN_BASELINE_MAX/100:
        range_size = HUMAN_BASELINE_MAX/100 - HUMAN_BASELINE_TYPICAL/100
        position = (composite_score - HUMAN_BASELINE_TYPICAL/100) / range_size
        likelihood = 4.5 + (position * 0.25)
    else:
        range_size = 1.0 - HUMAN_BASELINE_MAX/100
        position = (composite_score - HUMAN_BASELINE_MAX/100) / range_size
        likelihood = 4.75 + (position * 0.25)
    
    return max(0, min(5, likelihood))

def calculate_record_likelihood(row, resource_data, role_to_group, group_roles, weights):
    """Calculate likelihood score for a single record"""
    
    classification = row.get('major_role_group', '')
    sub_group = row.get('minor_sub_group', '')
    justification = row.get('grouping_justification', '')
    job_title = row.get('new_job_title', row.get('job_title_original', 'Unknown'))
    
    similarity_score, similarity_reason = calculate_similarity_score(
        classification, role_to_group, group_roles
    )
    
    salary_score, salary_reason, salary_amount = calculate_salary_impact(
        classification, resource_data['salary_data']
    )
    
    avg_correction_time = resource_data['time_to_correct'].iloc[0]['Average']
    time_score = 1.0 - (avg_correction_time / 80)
    
    subgroup_score = 0.8 if sub_group in ['I', 'II', 'III'] else 0.5
    
    justification_score = min(1.0, len(str(justification)) / 500) if justification else 0.0
    
    composite_score = (
        weights['similarity'] * similarity_score +
        weights['salary'] * salary_score +
        weights['time'] * time_score +
        weights['subgroup'] * subgroup_score +
        weights['justification'] * justification_score
    )
    
    likelihood = calculate_likelihood_from_composite(composite_score)
    
    return {
        'job_title': job_title,
        'classification': classification,
        'sub_group': sub_group,
        'likelihood_score': likelihood,
        'accuracy_equivalent': composite_score * 100,
        'similarity_score': similarity_score,
        'similarity_reason': similarity_reason,
        'salary_score': salary_score,
        'salary_reason': salary_reason,
        'salary_amount': salary_amount,
        'time_score': time_score,
        'correction_hours': avg_correction_time,
        'subgroup_score': subgroup_score,
        'justification_score': justification_score,
        'justification_length': len(str(justification)),
        'composite_score': composite_score
    }

print("✅ Core calculation functions loaded")

In [ ]:
#@title 6️⃣ Process All Records and Calculate Individual Scores { display-mode: "form" }
#@markdown This will calculate likelihood scores for EACH record in your batch

# Preprocess data
print("Processing data...")
resource_data['salary_data'] = preprocess_salary_data(resource_data['salary_data'])
role_to_group, group_roles, group_details = create_role_mappings(resource_data['role_groups'])

# Calculate scores for each record
results = []
for idx, row in input_data['classifications'].iterrows():
    record_result = calculate_record_likelihood(
        row, resource_data, role_to_group, group_roles, weights
    )
    record_result['record_id'] = idx
    record_result['source_row'] = row.get('source_row_index', idx)
    results.append(record_result)
    
    if (idx + 1) % 10 == 0:
        print(f"  Processed {idx + 1}/{len(input_data['classifications'])} records...")

# Create results DataFrame
results_df = pd.DataFrame(results)

print(f"\n✅ Processed {len(results_df)} records successfully!")

# Calculate summary statistics
print("\n📊 SUMMARY STATISTICS:")
print("=" * 80)
print(f"Overall Mean Likelihood: {results_df['likelihood_score'].mean():.2f} / 5.00")
print(f"Overall Accuracy Equivalent: {results_df['accuracy_equivalent'].mean():.1f}%")
print(f"Median Likelihood: {results_df['likelihood_score'].median():.2f}")
print(f"Std Dev: {results_df['likelihood_score'].std():.2f}")
print(f"Range: {results_df['likelihood_score'].min():.2f} - {results_df['likelihood_score'].max():.2f}")

In [ ]:
#@title 7️⃣ View Individual Record Scores { display-mode: "form" }

# Show all records with their individual scores
print("\n📋 ALL INDIVIDUAL RECORD SCORES:")
print("=" * 80)

# Sort by likelihood score
sorted_results = results_df.sort_values('likelihood_score', ascending=False)

# Display key columns
display_df = sorted_results[[
    'record_id', 
    'job_title', 
    'classification', 
    'likelihood_score', 
    'accuracy_equivalent'
]].copy()

display_df['likelihood_score'] = display_df['likelihood_score'].round(2)
display_df['accuracy_equivalent'] = display_df['accuracy_equivalent'].round(1)

print(display_df.to_string(index=False))

print(f"\n✅ Showing all {len(results_df)} records sorted by likelihood score")

In [ ]:
#@title 8️⃣ Visualize Results { display-mode: "form" }

# Create visualizations
plt.style.use('seaborn-v0_8-darkgrid')
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# 1. Individual Score Distribution
ax1 = axes[0, 0]
ax1.hist(results_df['likelihood_score'], bins=20, color='skyblue', edgecolor='black', alpha=0.7)
ax1.axvline(4.0, color='green', linestyle='--', label='Human Min (4.0)')
ax1.axvline(results_df['likelihood_score'].mean(), color='red', linestyle='--', label=f'Mean: {results_df["likelihood_score"].mean():.2f}')
ax1.set_xlabel('Likelihood Score')
ax1.set_ylabel('Number of Records')
ax1.set_title('Distribution of Individual Likelihood Scores')
ax1.legend()

# 2. Scatter plot of all records
ax2 = axes[0, 1]
colors = ['green' if x >= 4.0 else 'orange' if x >= 3.0 else 'red' for x in results_df['likelihood_score']]
ax2.scatter(range(len(results_df)), results_df['likelihood_score'], c=colors, alpha=0.6)
ax2.axhline(y=4.0, color='green', linestyle='--', alpha=0.5, label='Human Minimum')
ax2.set_xlabel('Record Index')
ax2.set_ylabel('Likelihood Score')
ax2.set_title('Individual Record Scores')
ax2.legend()

# 3. Component scores for all records
ax3 = axes[1, 0]
component_means = [
    results_df['similarity_score'].mean(),
    results_df['salary_score'].mean(),
    results_df['time_score'].mean(),
    results_df['subgroup_score'].mean(),
    results_df['justification_score'].mean()
]
components = ['KSAC', 'Salary', 'Time', 'Subgroup', 'Justification']
bars = ax3.bar(components, component_means, color=['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4', '#FFEAA7'])
ax3.set_ylabel('Average Score')
ax3.set_title('Average Component Scores Across All Records')
ax3.set_ylim(0, 1)
for bar, score in zip(bars, component_means):
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
             f'{score:.2f}', ha='center', va='bottom')

# 4. Performance categories
ax4 = axes[1, 1]
def categorize_performance(score):
    if score >= 4.75: return "Superhuman"
    elif score >= 4.5: return "Excellent" 
    elif score >= 4.0: return "Human-level"
    elif score >= 3.0: return "Below Human"
    elif score >= 2.0: return "Poor"
    else: return "Critical"

results_df['category'] = results_df['likelihood_score'].apply(categorize_performance)
category_counts = results_df['category'].value_counts()
colors_pie = ['#2ecc71', '#3498db', '#9b59b6', '#f39c12', '#e67e22', '#e74c3c']
ax4.pie(category_counts.values, labels=category_counts.index, autopct='%1.1f%%', colors=colors_pie[:len(category_counts)])
ax4.set_title('Performance Category Distribution')

plt.tight_layout()
plt.show()

print("\n✅ Visualizations generated!")

In [ ]:
#@title 9️⃣ Export Results with Individual Scores { display-mode: "form" }

# Prepare comprehensive export
export_df = results_df[[
    'record_id',
    'source_row',
    'job_title',
    'classification',
    'sub_group',
    'likelihood_score',
    'accuracy_equivalent',
    'category',
    'similarity_score',
    'similarity_reason',
    'salary_score',
    'salary_reason',
    'time_score',
    'correction_hours',
    'subgroup_score',
    'justification_score'
]].copy()

# Round for clarity
numeric_cols = ['likelihood_score', 'accuracy_equivalent', 'similarity_score', 
                'salary_score', 'time_score', 'subgroup_score', 'justification_score']
for col in numeric_cols:
    if col in export_df.columns:
        export_df[col] = export_df[col].round(2)

# Save files
export_filename = 'individual_likelihood_scores.csv'
export_df.to_csv(export_filename, index=False)

print(f"✅ Exported {len(export_df)} records with individual scores to {export_filename}")
print("\n📥 Downloading file...")
files.download(export_filename)

print("\n🎉 Complete! Each record now has its individual likelihood score.")